# Energy-Preserving Operator Inference: Viscous Burgers' Equation

In this notebook, we conduct the numerical experiment for energy-preserving Operator Inference (EP-OpInf). The standard (least-squares) Operator Inference problem is augmented so that the learned quadratic operator $\hat{\mathbf{F}}$ is (approximately) energy preserving, i.e., $\hat{\mathbf{x}}^\top\hat{\mathbf{F}}(\hat{\mathbf{x}}\otimes\hat{\mathbf{x}}) = 0$ for all $\hat{\mathbf{x}}$. Three variants are compared:

- **EPHEC-OpInf**: energy-preserving *hard equality constraints*,
- **EPSIC-OpInf**: energy-preserving *soft inequality constraints* $|\cdot| \le \epsilon$,
- **EPP-OpInf**: an energy-preserving *penalty* $\alpha\,\|\cdot\|^2$ added to the objective,

against standard least-squares OpInf and the intrusive (POD-Galerkin) reduced model. The test problem is the viscous Burgers' equation on a periodic domain
$$
\frac{\partial x}{\partial t} = \mu\frac{\partial^2 x}{\partial \omega^2} - x\frac{\partial x}{\partial \omega}, \qquad \omega \in [0, 1), \quad t \in [0, 1],
$$
with $\mu = 0.1$, discretized by central finite differences ($N = 128$ grid points) and integrated with a semi-implicit Euler scheme ($\Delta t = 10^{-4}$).

The workflow is: generate training/test data → compute the reduced operators (least squares, intrusive, EPHEC, EPSIC, EPP) → analyze the errors and the energy-preserving constraint → save the results to `data/`.

The EP-OpInf problems are solved with [JuMP](https://jump.dev) and [Ipopt](https://coin-or.github.io/Ipopt/).

## Packages

In [ ]:
using DataFrames
using JLD2
using LaTeXStrings
using LinearAlgebra
using Plots
using Plots.PlotMeasures
using ProgressMeter
using Random
using Statistics
using UniqueKronecker
import HSL_jll
using PolynomialModelReductionDataset: BurgersModel
using LiftAndLearn
const LnL = LiftAndLearn
;

## Experiment Size

The full experiment of the paper uses 75 training initial conditions and 50 random test initial conditions (inside and outside the training region). The default demo setting uses the same training data, reduced dimensions, and solver settings, but only 10 test initial conditions per test; set `PAPER_SCALE = true` to reproduce the paper setting. The test averages therefore differ somewhat between the two settings, but the conclusions are the same. The test-data generation and the analysis loops use `Threads.@threads`, so starting Julia with several threads speeds them up.

In [ ]:
# Set to `true` to reproduce the full-size experiments from the paper (about 6 minutes
# single-threaded); the default demo settings run in about 4 minutes.
const PAPER_SCALE = false

# Number of random test initial conditions (for each of test 1 and test 2; must be even)
num_test_ic = PAPER_SCALE ? 50 : 10
;

## Initial Setup

In [ ]:
# First order Burger's equation setup
Ω = (0.0, 1.0)
Nx = 2^7; dt = 1e-4
burger = BurgersModel(
    spatial_domain=Ω, time_domain=(0.0, 1.0), Δx=(Ω[2] - Ω[1])/Nx, Δt=dt,
    diffusion_coeffs=0.1, BC=:periodic,
)
;

In [ ]:
# File to save the results to
mkpath("data")
filename = "data/epOpInf_burgers_results.jld2"
;

### Important Parameters

In [ ]:
# Use HSL's ma97 if a licensed HSL library is installed; otherwise Ipopt's default (MUMPS)
const USE_HSL = ccall((:LIBHSL_isfunctional, HSL_jll.libhsl), Bool, ())

# Minimum and maximum reduced dimensions
rmin = 1
rmax = 10

burger_system = LnL.SystemStructure(
    state=[1,2],  # linear and quadratic state operators
)
burger_vars = LnL.VariableStructure(
    N=1,
)
burger_data = LnL.DataStructure(
    Δt=dt,
    DS=100,
)
burger_optim = LnL.OptimizationSetting(
    verbose=false,
    initial_guess=false,
    max_iter=1000,
    reproject=false,
    SIGE=false,
    linear_solver = USE_HSL ? "ma97" : "none",
    HSL_lib_path = USE_HSL ? HSL_jll.libhsl_path : "none",
)

# Downsampling rate
DS = burger_data.DS

# Downsampled time grid (the data and errors are evaluated on this grid)
t_ds = burger.tspan[1:DS:end]
;

### Initial Conditions

Three families of parameterized initial conditions are available. The paper (and the default `IC_FAMILY = :sine`) uses a sinusoidal wave with three parameters: amplitude ($a$), frequency ($b$), and phase ($c$)

$$
\begin{gather*}
    x(\omega, 0) = a\sin(2\pi \lceil b \rceil \omega + c)
\end{gather*}
$$

`IC_FAMILY = :exp_cos` uses a combination of exponential and cosine:

$$
\begin{gather*}
    x(\omega, 0) = \exp(-a\cos^2(\pi \omega+b)) - c
\end{gather*}
$$

`IC_FAMILY = :exp_mixed` uses a complex wave with a combination of exponential, cosine, and sine waves:

$$
\begin{gather*}
    x(\omega, 0) = \exp\left(-a\cos^2(\pi\omega+b) + \sin^2\left(4\pi\omega+\frac{\pi}{3}\right) - \cos^2(3\pi\omega) \right)+c
\end{gather*}
$$

In all cases, $a, b, c$ are the three parameters to adjust. The training data uses all combinations of the training parameter values (`ic_a`, `ic_b`, `ic_c`); the parameter ranges `ic_*_out` are used to sample test initial conditions outside of the training region.

In [ ]:
# Family of initial conditions (:sine, :exp_cos, or :exp_mixed)
IC_FAMILY = :sine

if IC_FAMILY == :sine
    # Training parameter values
    ic_a = range(0.8, 1.2, length=5)
    ic_b = [1, 2, 3]
    ic_c = range(-0.25, 0.25, length=5)
    # Parameter domains for test 2 (outside the training region)
    ic_a_out = [0.5, 1.5]
    ic_b_out = [3, 6]
    ic_c_out = [-0.5, 0.5]
    # Parameterized initial condition
    init_wave = (a, b, c) -> a .* sin.(2π * ceil(b) .* burger.xspan .+ c)
elseif IC_FAMILY == :exp_cos
    ic_a = [2, 4, 6]
    ic_b = [-0.5, 0, 0.5]
    ic_c = [-0.25, 0, 0.25]
    ic_a_out = [0.1, 8]
    ic_b_out = [-0.8, 0.8]
    ic_c_out = [-0.5, 0.5]
    init_wave = (a, b, c) -> exp.(-a * cos.(π .* burger.xspan .+ b).^2) .- c
elseif IC_FAMILY == :exp_mixed
    ic_a = [1, 5, 9]
    ic_b = [-π/4, 0, π/4]
    ic_c = [-0.5, 0, 0.5]
    ic_a_out = [0.01, 20]
    ic_b_out = [-π/2, π/2]
    ic_c_out = [-1.0, 1.0]
    init_wave = (a, b, c) -> exp.(-a * cos.(π .* burger.xspan .+ b).^2 .+ sin.(4π .* burger.xspan .+ π/3).^2 .- cos.(3π .* burger.xspan).^2) .+ c
else
    error("Unknown IC_FAMILY = $(IC_FAMILY)")
end

num_ic_params = length(ic_a) * length(ic_b) * length(ic_c)
@info "Number of training initial conditions: $(num_ic_params)"
;

## Generate Training Data

For every training initial condition, the full model is integrated over $t \in [0,1]$ with $\Delta t = 10^{-4}$. The snapshots and the finite-difference time derivatives are downsampled by `DS = 100` and stacked into one training data matrix, from which the POD basis is computed. To save memory, only the downsampled trajectories are stored for the analysis (plus one full trajectory for plotting).

In [ ]:
# Store values
Xtr = Vector{Matrix{Float64}}(undef, burger.param_dim)  # training state data
Rtr = Vector{Matrix{Float64}}(undef, burger.param_dim)  # training derivative data
Σr = Vector{Vector{Float64}}(undef, burger.param_dim)  # singular values
Xtr_all = Matrix{Matrix{Float64}}(undef, burger.param_dim, num_ic_params)  # all (downsampled) training trajectories
IC_train = Vector{Vector{Float64}}(undef, num_ic_params)  # all initial conditions
Vrmax = Vector{Matrix{Float64}}(undef, burger.param_dim)  # POD basis
op_fom_tr = Vector{LnL.Operators}(undef, burger.param_dim)  # FOM operators
X_example = Matrix{Float64}(undef, 0, 0)  # one full trajectory for plotting

@info "Generate the FOM system matrices and training data."
for i in 1:burger.param_dim
    μ = burger.diffusion_coeffs[i]

    # Generate the FOM system matrices (ONLY DEPENDS ON μ)
    A, F = burger.finite_diff_model(burger, μ)
    op_fom_tr[i] = LnL.Operators(A=A, A2u=F)

    # Store the training data
    Xall = Vector{Matrix{Float64}}(undef, num_ic_params)
    Xdotall = Vector{Matrix{Float64}}(undef, num_ic_params)

    # Generate the data for all combinations of the initial condition parameters
    prog = Progress(num_ic_params; desc="Training data (μ=$(μ)): ")
    ct = 1  # set/reset counter
    for a in ic_a, b in ic_b, c in ic_c
        if i == 1
            IC_train[ct] = init_wave(a, b, c)
        end

        states = burger.integrate_model(burger.tspan, IC_train[ct]; linear_matrix=A, quadratic_matrix=F, system_input=false)
        Xtr_all[i,ct] = states[:, 1:DS:end]  # downsampled trajectory (including t = 0)
        if i == 1 && ct == 1
            global X_example = states
        end

        tmp = states[:, 2:end]
        Xall[ct] = tmp[:, 1:DS:end]  # downsample data
        tmp = (states[:, 2:end] - states[:, 1:end-1]) / burger.Δt
        Xdotall[ct] = tmp[:, 1:DS:end]  # downsample data

        ct += 1  # increment counter
        next!(prog)
    end
    # Combine all initial condition data to form on big training data matrix
    Xtr[i] = reduce(hcat, Xall)
    Rtr[i] = reduce(hcat, Xdotall)

    # Compute the POD basis from the training data
    tmp = svd(Xtr[i])
    Vrmax[i] = tmp.U[:, 1:rmax]
    Σr[i] = tmp.S
end
@info "Training data matrix size: $(size(Xtr[1]))"

In [ ]:
p1 = plot()
timestamps = 1:1000:burger.time_dim
plot!(
    p1, burger.xspan, X_example[:, timestamps], color=:roma, line_z=(burger.Δt*(timestamps .- 1))',
    lw=1.5, label="", colorbar_ticks=0:0.1:1.0, clims=(0,1.0),
)
plot!(p1, xlabel=L"\omega"*", spatial coordinate", ylabel=L"x(\omega,t)", grid=true, minorgrid=true)
plot!(p1, guidefontsize=16, tickfontsize=13,  legendfontsize=13, fontfamily="Computer Modern")
plot!(p1, colorbar_title=L"t", colorbar_titlefontsize=16)

In [ ]:
tplot = 1:10:burger.time_dim  # plot every 10th time step
plot(contourf(burger.tspan[tplot], burger.xspan, X_example[:, tplot], lw=0), xlabel=L"t", ylim=[0, 1.0],
    ylabel=L"\omega, \mathrm{spatial~coordinate}", zlabel=L"x(\omega,t)")
yticks!(0:0.2:1.0)
plot!(guidefontsize=13, fontfamily="Computer Modern")

The next two cells create high-resolution figures with a transparent background (e.g., for presentation slides) and save them to `figures/`.

In [ ]:
mkpath("figures")
plot(contourf(burger.tspan[tplot], burger.xspan, X_example[:, tplot], lw=0, color=:inferno), xlabel=L"t", ylim=[0, 1.0],
    ylabel=L"\omega, \mathrm{spatial~coordinate}", zlabel=L"x(\omega,t)", size=(1200, 800))
yticks!(0:0.2:1.0)
plot!(guidefontsize=32, tickfontsize=25,  legendfontsize=13, fontfamily="Computer Modern", right_margin=13mm, left_margin=6mm, bottom_margin=4mm, top_margin=3mm)
plot!(background_color=:transparent, background_color_inside="#44546A", dpi=600)
savefig("figures/burgers_contour.png");

In [ ]:
mkpath("figures")
tsurf = 1:50:burger.time_dim  # coarser time grid for the surface plot
plot(surface(burger.tspan[tsurf], burger.xspan, X_example[:, tsurf], lw=0, color=:inferno), xlabel=L"t", ylim=[0, 1.0],
    ylabel=L"\omega", zlabel=L"x(\omega,t)", camera=(-50,20), colorbar=false, showaxis=false, grid=false)
plot!(background_color=:transparent, dpi=600)
savefig("figures/burgers_surface.png");

### Check Energy-Levels from SVD and Decide Reduced Orders

In [ ]:
nice_orders_all = Vector{Vector{Int}}(undef, burger.param_dim)
energy_level_all = Vector{Vector{Float64}}(undef, burger.param_dim)

for i in eachindex(Σr)
    nice_orders_all[i], energy_level_all[i] = LnL.choose_ro(Σr[i]; en_low=-12)
end
nice_orders = Int.(round.(mean(nice_orders_all)))
energy_level = mean(energy_level_all)
@info "Reduced orders reaching energy losses 1e-1, 1e-2, ..., 1e-12: $(nice_orders)"

In [ ]:
plot(energy_level[1:nice_orders[end]], yaxis=:log10, label="energy loss", fontfamily="Computer Modern", lw=2,
    ylabel="relative energy loss from truncation", xlabel="number of retained modes", legend=:topright, majorgrid=true, grid=true)
plot!(nice_orders, energy_level[nice_orders], seriestype=:scatter, label=nothing, markersize=6)
for i in 1:length(nice_orders)
    if i == 7
        annotate!((nice_orders[i]+2.5, energy_level[nice_orders[i]], text(L"r_{\mathrm{max}}="*string(nice_orders[i]), :bottom, 18, "Computer Modern")))
    end
end
yticks!(10.0 .^ (0:-2:-10))
xticks!(0:2:20)
plot!(fg_legend=:false)
plot!(guidefontsize=14, tickfontsize=12, titlefontsize=15,  legendfontsize=15, right_margin=2mm)

## Generate Test Data (Inside Training Region)

`num_test_ic` random amplitudes, frequencies, and phases from the training region (50 in the paper).

In [ ]:
# Store values
Xtest_in = Matrix{Matrix{Float64}}(undef, burger.param_dim, num_test_ic)  # all (downsampled) test trajectories
IC_test_in = Vector{Vector{Float64}}(undef, num_test_ic)  # all initial conditions

@info "Generate test data for parameters within the training region."

# Random initial condition parameters inside the training region
rng = MersenneTwister(1)
ic_a_rand_in = (maximum(ic_a) - minimum(ic_a)) .* rand(rng, num_test_ic) .+ minimum(ic_a)
ic_b_rand_in = (maximum(ic_b) - minimum(ic_b)) .* rand(rng, num_test_ic) .+ minimum(ic_b)
ic_c_rand_in = (maximum(ic_c) - minimum(ic_c)) .* rand(rng, num_test_ic) .+ minimum(ic_c)

for j in 1:num_test_ic
    IC_test_in[j] = init_wave(ic_a_rand_in[j], ic_b_rand_in[j], ic_c_rand_in[j])
end

for i in 1:burger.param_dim
    A = op_fom_tr[i].A
    F = op_fom_tr[i].A2u

    # Integrate the full model for all test initial conditions
    prog = Progress(num_test_ic; desc="Test 1 data: ")
    Threads.@threads for j in 1:num_test_ic
        states = burger.integrate_model(burger.tspan, IC_test_in[j]; linear_matrix=A, quadratic_matrix=F, system_input=false)
        Xtest_in[i,j] = states[:, 1:DS:end]
        next!(prog)
    end
end

## Generate Test Data (Outside Training Region)

`num_test_ic` random amplitudes, frequencies, and phases outside of the training region (50 in the paper). Half of the amplitudes/phases are below and half are above the training range; the frequencies are sampled from `ic_b_out`.

In [ ]:
# Store values
Xtest_out = Matrix{Matrix{Float64}}(undef, burger.param_dim, num_test_ic)  # all (downsampled) test trajectories
IC_test_out = Vector{Vector{Float64}}(undef, num_test_ic)  # all initial conditions

@info "Generate test data for parameters outside the training region."

# Random initial condition parameters outside the training region
rng = MersenneTwister(2)
ϵ = 1e-2
half_num_test_ic = num_test_ic ÷ 2
ic_a_rand_out = ((minimum(ic_a) - ϵ) - ic_a_out[1]) .* rand(rng, half_num_test_ic) .+ ic_a_out[1]
append!(ic_a_rand_out, (ic_a_out[2] - (maximum(ic_a) + ϵ)) .* rand(rng, num_test_ic - half_num_test_ic) .+ (maximum(ic_a) + ϵ))
ic_b_rand_out = (ic_b_out[2] - ic_b_out[1] + ϵ) .* rand(rng, num_test_ic) .+ ic_b_out[1]
ic_c_rand_out = ((minimum(ic_c) - ϵ) - ic_c_out[1]) .* rand(rng, half_num_test_ic) .+ ic_c_out[1]
append!(ic_c_rand_out, (ic_c_out[2] - (maximum(ic_c) + ϵ)) .* rand(rng, num_test_ic - half_num_test_ic) .+ (maximum(ic_c) + ϵ))

for j in 1:num_test_ic
    IC_test_out[j] = init_wave(ic_a_rand_out[j], ic_b_rand_out[j], ic_c_rand_out[j])
end

for i in 1:burger.param_dim
    A = op_fom_tr[i].A
    F = op_fom_tr[i].A2u

    # Integrate the full model for all test initial conditions
    prog = Progress(num_test_ic; desc="Test 2 data: ")
    Threads.@threads for j in 1:num_test_ic
        states = burger.integrate_model(burger.tspan, IC_test_out[j]; linear_matrix=A, quadratic_matrix=F, system_input=false)
        Xtest_out[i,j] = states[:, 1:DS:end]
        next!(prog)
    end
end

## Obtain OpInf Least-Squares Operators

The operators are inferred once with the largest basis $\mathbf{V}_{r_{\max}}$; the operators of the smaller models are obtained by extracting the leading blocks.

In [ ]:
@info "Compute the Least Squares solution."

options = LnL.LSOpInfOption(
    system=burger_system,
    vars=burger_vars,
    data=burger_data,
    optim=burger_optim,
)

op_LS = Vector{LnL.Operators}(undef, burger.param_dim)
for i in 1:burger.param_dim
    op_LS[i] = LnL.opinf(Xtr[i], Vrmax[i], options; Xdot=Rtr[i])
end

## Obtain intrusive operators

In [ ]:
@info "Compute the intrusive model"

op_int = Vector{LnL.Operators}(undef, burger.param_dim)
for i in 1:burger.param_dim
    # Project the full-model operators onto the POD basis of the training data
    op_int[i] = LnL.pod(op_fom_tr[i], Vrmax[i], burger_system)
end

## Obtain operators for EP-OpInf

Each EP-OpInf variant is a constrained (or penalized) least-squares problem solved by Ipopt. Ipopt uses HSL's MA97 linear solver if a licensed HSL library is available (`USE_HSL`), and its default MUMPS solver otherwise.

After each solve, LiftAndLearn logs a summary (`@info`) with the solve time, Ipopt's termination status, and the objective value; set `burger_optim.verbose = true` to also see Ipopt's full iteration log. We additionally check each result by computing the energy-preserving constraint residual
$$
\epsilon_{\mathrm{EP}} = \sum_{i,j,k=1}^{r} \left|\hat{h}_{ijk} + \hat{h}_{jik} + \hat{h}_{kji}\right|,
$$
where $\hat{h}_{ijk}$ are the entries of the quadratic operator $\hat{\mathbf{H}}$ (with $\hat{\mathbf{H}}(\hat{\mathbf{x}}\otimes\hat{\mathbf{x}}) = \hat{\mathbf{F}}(\hat{\mathbf{x}}\oslash\hat{\mathbf{x}})$).

In [ ]:
# Energy-preserving constraint residual of the quadratic operator of `op`
ep_residual(op::LnL.Operators) = LnL.ep_constraint_residual(op.A2u, size(op.A2u, 1))
;

### EPHEC-OpInf (hard equality constraints)

In [ ]:
@info "Compute the EPHEC model"

options = LnL.EPHECOpInfOption(
    system=burger_system,
    vars=burger_vars,
    data=burger_data,
    optim=burger_optim,
)

op_ephec = Vector{LnL.Operators}(undef, burger.param_dim)
for i in 1:burger.param_dim
    options.optim.initial_guess = false  # use Ipopt's default initial guess
    t_solve = @elapsed op_ephec[i] = LnL.epopinf(Xtr[i], Vrmax[i], options; Xdot=Rtr[i])
    @info "EPHEC (μ = $(burger.diffusion_coeffs[i])): $(round(t_solve, digits=1)) s, EP residual = $(ep_residual(op_ephec[i])) (LS OpInf: $(ep_residual(op_LS[i])))"
end

### EPSIC-OpInf (soft inequality constraints)

In [ ]:
@info "Compute the EPSIC OpInf."

options = LnL.EPSICOpInfOption(
    system=burger_system,
    vars=burger_vars,
    data=burger_data,
    optim=burger_optim,
    ϵ=1e-3,
)

op_epsic = Vector{LnL.Operators}(undef, burger.param_dim)
for i in 1:burger.param_dim
    options.optim.initial_guess = false  # use Ipopt's default initial guess
    t_solve = @elapsed op_epsic[i] = LnL.epopinf(Xtr[i], Vrmax[i], options; Xdot=Rtr[i])
    @info "EPSIC (μ = $(burger.diffusion_coeffs[i])): $(round(t_solve, digits=1)) s, EP residual = $(ep_residual(op_epsic[i])) (LS OpInf: $(ep_residual(op_LS[i])))"
end

### EPP-OpInf (penalty)

A large penalty weight $\alpha$ drives the constraint residual down but makes the problem badly scaled. With $\alpha = 10^8$, Ipopt stops at an "acceptable" rather than optimal solution for the default initial conditions (termination status `ALMOST_LOCALLY_SOLVED`); with `IC_FAMILY = :exp_cos` it even runs into `max_iter = 1000` (`ITERATION_LIMIT`, taking several minutes), although the resulting operator is still usable (residual $\sim 10^{-7}$). A smaller weight (e.g., $\alpha = 10^6$) solves faster at the price of a larger constraint residual.

In [ ]:
@info "Compute the EPP OpInf."

options = LnL.EPPOpInfOption(
    system=burger_system,
    vars=burger_vars,
    data=burger_data,
    optim=burger_optim,
    α=1e8,
)

op_epp = Vector{LnL.Operators}(undef, burger.param_dim)
for i in 1:burger.param_dim
    options.optim.initial_guess = false  # use Ipopt's default initial guess
    t_solve = @elapsed op_epp[i] = LnL.epopinf(Xtr[i], Vrmax[i], options; Xdot=Rtr[i])
    @info "EPP (μ = $(burger.diffusion_coeffs[i])): $(round(t_solve, digits=1)) s, EP residual = $(ep_residual(op_epp[i])) (LS OpInf: $(ep_residual(op_LS[i])))"
end

## Analyze the Reduced Models

For each reduced dimension $r = r_{\min}, \ldots, r_{\max}$, the operators of all five models are truncated to dimension $r$ and the reduced models are integrated from the projected initial conditions with the same semi-implicit Euler scheme. We compute

- the projection error and the relative state error $\|\mathbf{X} - \mathbf{V}_r\hat{\mathbf{X}}\|_F / \|\mathbf{X}\|_F$ (on the downsampled time grid), averaged over the initial conditions,
- the energy-preserving constraint residual $\epsilon_{\mathrm{EP}}$ and the "momentum" $\sum_{i,j,k}(\hat{h}_{ijk} + \hat{h}_{jik} + \hat{h}_{kji})$ of the quadratic operator,
- the energy $\frac{1}{2}\|\hat{\mathbf{x}}(t)\|^2$ and the constraint violation $\hat{\mathbf{x}}(t)^\top\hat{\mathbf{F}}(\hat{\mathbf{x}}(t)\oslash\hat{\mathbf{x}}(t))$ (the rate of change of the energy caused by the quadratic term) along the trajectories, averaged over the initial conditions.

Note that the finite-difference discretization of the non-conservative form $x\,\partial x / \partial \omega$ used here is *not* exactly energy preserving, so neither the full model nor the intrusive model satisfies the constraint exactly.

In [ ]:
const METHODS = [:int, :LS, :ephec, :epsic, :epp]
const METHOD_NAMES = Dict(:int => "Intrusive", :LS => "OpInf", :ephec => "EPHEC-OpInf",
                          :epsic => "EPSIC-OpInf", :epp => "EPP-OpInf")

"""
    ep_analyze(model, ops, op_fom, Vrmax, X_all, IC, rmin, rmax, DS)

Integrate the reduced models in `ops` (a Dict `method => Vector{Operators}`) for every initial
condition in `IC` and reduced dimension `r = rmin:rmax`, and compare them with the
(downsampled) full-model trajectories `X_all[i, ct]`.
"""
function ep_analyze(model, ops, op_fom, Vrmax, X_all, IC, rmin, rmax, DS)
    nr = rmax - rmin + 1
    np = model.param_dim
    nt = size(X_all[1, 1], 2)
    nic = length(IC)

    PE = zeros(nr, np)                                 # projection error
    SE = Dict(m => zeros(nr, np) for m in METHODS)     # relative state error
    CR = Dict(m => zeros(nr, np) for m in METHODS)     # EP constraint residual
    MMT = Dict(m => zeros(nr, np) for m in METHODS)    # momentum
    EN = Dict(m => zeros(nt, nr, np) for m in METHODS) # energy over time
    CV = Dict(m => zeros(nt, nr, np) for m in METHODS) # constraint violation over time
    EN_fom = zeros(nt, np)
    CV_fom = zeros(nt, np)

    prog = Progress(np * nr; desc="Analyze: ")
    for i in 1:np
        # Energy and constraint violation of the full model
        F_full = op_fom[i].A2u
        EN_fom[:, i] = mean(0.5 .* vec(sum(abs2, X_all[i, ct]; dims=1)) for ct in 1:nic)
        CV_fom[:, i] = mean(vec(LnL.ep_constraint_violation(X_all[i, ct], F_full)) for ct in 1:nic)

        Threads.@threads for j in 1:nr
            r = rmin + j - 1
            Vr = Vrmax[i][:, 1:r]
            PE[j, i] = mean(LnL.proj_error(X_all[i, ct], Vr) for ct in 1:nic)

            for m in METHODS
                # Truncate the operators to dimension r
                Ar = ops[m][i].A[1:r, 1:r]
                Fr = UniqueKronecker.extractF(ops[m][i].A2u, r)
                CR[m][j, i], MMT[m][j, i] = LnL.ep_constraint_residual(Fr, r; with_moment=true)

                se = zeros(nic)
                for (ct, ic) in enumerate(IC)
                    Xr = model.integrate_model(model.tspan, Vr' * ic; linear_matrix=Ar, quadratic_matrix=Fr, system_input=false)
                    Xr = Xr[:, 1:DS:end]
                    se[ct] = LnL.rel_state_error(X_all[i, ct], Xr, Vr)
                    EN[m][:, j, i] .+= 0.5 .* vec(sum(abs2, Xr; dims=1)) ./ nic
                    CV[m][:, j, i] .+= vec(LnL.ep_constraint_violation(Xr, Fr)) ./ nic
                end
                SE[m][j, i] = mean(se)
            end
            next!(prog)
        end
    end
    return (; PE, SE, CR, MMT, EN, CV, EN_fom, CV_fom)
end

ops = Dict(:int => op_int, :LS => op_LS, :ephec => op_ephec, :epsic => op_epsic, :epp => op_epp)

# Summary table: values (averaged over the parameters μ) for each reduced dimension
function results_table(res, field; rs=rmin:rmax)
    df = DataFrame(r = collect(rs))
    for m in METHODS
        df[!, METHOD_NAMES[m]] = vec(mean(getfield(res, field)[m], dims=2))
    end
    return df
end
;

## Analyze Training Results

In [ ]:
train_res = ep_analyze(burger, ops, op_fom_tr, Vrmax, Xtr_all, IC_train, rmin, rmax, DS)
;

### Results
#### Projection Error

In [ ]:
# Training data
mean_train_proj_err = mean(train_res.PE, dims=2)
plot(rmin:rmax, mean_train_proj_err, marker=(:rect))
plot!(yscale=:log10, majorgrid=true, legend=false)
tmp = log10.(mean_train_proj_err)
yticks!([10.0^i for i in floor(minimum(tmp))-1:ceil(maximum(tmp))+1])
xticks!(rmin:rmax)
xlabel!(L"\mathrm{reduced~model~dimension~} r")
ylabel!(L"\mathrm{average~~projection~~error}")
plot!(guidefontsize=16, tickfontsize=13,  legendfontsize=13)

#### Mean relative state error

In [ ]:
results_table(train_res, :SE)

In [ ]:
# Plot the average relative state errors of all models
function plot_state_errors!(p, SE; labels=true)
    lab(m) = labels ? METHOD_NAMES[m] : ""
    mean_err = Dict(m => vec(mean(SE[m], dims=2)) for m in METHODS)
    plot!(p, rmin:rmax, mean_err[:int], c=:orange, marker=(:cross, 10), markerstrokewidth=2.5, label=lab(:int))
    plot!(p, rmin:rmax, mean_err[:LS], c=:crimson, marker=(:circle, 5), markerstrokecolor=:crimson, lw=2, label=lab(:LS))
    plot!(p, rmin:rmax, mean_err[:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label=lab(:ephec))
    plot!(p, rmin:rmax, mean_err[:epsic], c=:purple, markerstrokecolor=:purple, marker=(:dtriangle, 5), ls=:dot, label=lab(:epsic))
    plot!(p, rmin:rmax, mean_err[:epp], c=:brown, markerstrokecolor=:brown, marker=(:star, 5), lw=2, ls=:dashdot, label=lab(:epp))
    return p
end

p = plot_state_errors!(plot(), train_res.SE)
plot!(p, yscale=:log10, majorgrid=true, legend=:bottomleft)
xticks!(rmin:rmax)
xlabel!("reduced model dimension " * L"r")
ylabel!("average relative state error")
title!("Training")
plot!(guidefontsize=16, tickfontsize=13, legendfontsize=13, titlefontsize=18, fontfamily="Computer Modern")

#### Constraint Residual

In [ ]:
results_table(train_res, :CR)

In [ ]:
mean_CR = Dict(m => vec(mean(train_res.CR[m], dims=2)) for m in METHODS)

plot(rmin:rmax, mean_CR[:int], c=:orange, marker=(:cross, 10), markerstrokewidth=2.5, label="Intrusive")
plot!(rmin:rmax, mean_CR[:LS], marker=(:circle, 5), c=:crimson, label="OpInf", markerstrokecolor=:crimson, lw=2)
plot!(rmin:rmax, mean_CR[:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), lw=2, ls=:dash, label="EPHEC-OpInf")
plot!(rmin:rmax, mean_CR[:epsic], c=:purple, markerstrokecolor=:purple, marker=(:dtriangle, 5), ls=:dot, label="EPSIC-OpInf")
plot!(rmin:rmax, mean_CR[:epp], c=:brown, markerstrokecolor=:brown, marker=(:star, 5), ls=:dash, lw=2, label="EPP-OpInf")
plot!(yscale=:log10, majorgrid=true, legend=:bottomright, minorgridalpha=0.03)
yticks!(10.0 .^ (-30:5:5))
xticks!(rmin:rmax)
xlabel!("reduced model dimension " * L"r")
ylabel!("energy-preserving constraint residual")
plot!(xlabelfontsize=15, ylabelfontsize=13, tickfontsize=13,  legendfontsize=13, fontfamily="Computer Modern")

#### Momentum

In [ ]:
mean_mmt = Dict(m => abs.(vec(mean(train_res.MMT[m], dims=2))) for m in METHODS)

plot(rmin:rmax, mean_mmt[:int], c=:orange, marker=(:cross, 8), label="Intrusive")
plot!(rmin:rmax, mean_mmt[:LS], marker=(:circle, 3.5), c=:crimson, label="OpInf")
plot!(rmin:rmax, mean_mmt[:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), lw=2, ls=:dash, label="EPHEC-OpInf")
plot!(rmin:rmax, mean_mmt[:epsic], c=:purple, markerstrokecolor=:purple, marker=(:dtriangle, 5), ls=:dot, label="EPSIC-OpInf")
plot!(rmin:rmax, mean_mmt[:epp], c=:brown, markerstrokecolor=:brown, marker=(:star, 4), ls=:dash, lw=1, label="EPP-OpInf")
plot!(yscale=:log10, majorgrid=true, legend=:bottomright, minorgridalpha=0.03)
xticks!(rmin:rmax)
xlabel!("reduced model dimension " * L"r")
ylabel!("absolute value of momentum")
plot!(guidefontsize=16, tickfontsize=13, legendfontsize=10)

#### Energy and Constraint Violation over Time

The energy and the constraint violation along the trajectories of the reduced models of dimension `r_plot` (averaged over the initial conditions), compared with the full model.

In [ ]:
r_plot = rmax  # reduced dimension to show
jr = r_plot - rmin + 1

# Plot the average energy over time of the reduced models of dimension r_plot (and the full model)
function plot_energy(res; difference=false)
    EN = Dict(m => vec(mean(res.EN[m][:, jr, :], dims=2)) for m in METHODS)
    EN_fom = vec(mean(res.EN_fom, dims=2))
    ref = difference ? EN_fom : zero(EN_fom)
    p = plot(t_ds, EN[:int] - ref, lw=6, ls=:dash, c=:orange, label="Intrusive")
    plot!(p, t_ds, EN[:LS] - ref, lw=4, ls=:dash, c=:crimson, label="OpInf")
    plot!(p, t_ds, EN[:ephec] - ref, lw=3, ls=:dashdot, c=:blue, label="EPHEC-OpInf")
    plot!(p, t_ds, EN[:epsic] - ref, lw=3, ls=:dot, c=:purple, label="EPSIC-OpInf")
    plot!(p, t_ds, EN[:epp] - ref, lw=2, ls=:dashdot, c=:brown, label="EPP-OpInf")
    difference || plot!(p, t_ds, EN_fom, lw=1.5, c=:black, label="Full")
    difference || plot!(p, yscale=:log10)  # the energy decays by orders of magnitude
    plot!(p, majorgrid=true, minorgrid=true, legend=:topright)
    xlabel!(p, L"\mathrm{time}~t")
    ylabel!(p, difference ? "energy difference from full model" : "energy " * L"\frac{1}{2}\|x(t)\|^2")
    title!(p, (difference ? "Average energy difference" : "Average energy") * " (r = $(r_plot))")
    plot!(p, guidefontsize=14, tickfontsize=12, legendfontsize=11, titlefontsize=14)
    return p
end

plot_energy(train_res)

In [ ]:
plot_energy(train_res; difference=true)

In [ ]:
CV = Dict(m => abs.(vec(mean(train_res.CV[m][:, jr, :], dims=2))) for m in METHODS)
CV_fom = abs.(vec(mean(train_res.CV_fom, dims=2)))

# skip t = 0 (where the averaged violation can vanish) to plot on a log scale
it = 2:length(t_ds)
plot(t_ds[it], CV[:int][it], lw=4, ls=:dash, c=:orange, label="Intrusive")
plot!(t_ds[it], CV[:LS][it], lw=3, ls=:dash, c=:crimson, label="OpInf")
plot!(t_ds[it], CV[:ephec][it], lw=3, ls=:dashdot, c=:blue, label="EPHEC-OpInf")
plot!(t_ds[it], CV[:epsic][it], lw=3, ls=:dot, c=:purple, label="EPSIC-OpInf")
plot!(t_ds[it], CV[:epp][it], lw=2, ls=:dashdot, c=:brown, label="EPP-OpInf")
plot!(t_ds[it], CV_fom[it], lw=1.5, c=:black, label="Full")
plot!(yscale=:log10, majorgrid=true, legend=:outerright, size=(800, 400))
title!("Constraint violation (r = $(r_plot))")
xlabel!(L"\mathrm{time}~t")
ylabel!(L"|\hat{x}^\top \hat{F}(\hat{x} \oslash \hat{x})|")
plot!(guidefontsize=14, tickfontsize=11, legendfontsize=11, titlefontsize=14, left_margin=3mm, bottom_margin=3mm)

## Test 1: Inside Training Region

In [ ]:
test1_res = ep_analyze(burger, ops, op_fom_tr, Vrmax, Xtest_in, IC_test_in, rmin, rmax, DS)
results_table(test1_res, :SE)

### Results

In [ ]:
p = plot_state_errors!(plot(), test1_res.SE)
plot!(p, yscale=:log10, majorgrid=true, legend=:bottomleft)
xticks!(rmin:rmax)
xlabel!("reduced model dimension " * L"r")
ylabel!("average relative state error")
title!("Test 1 (inside training region)")
plot!(guidefontsize=16, tickfontsize=13, legendfontsize=13, titlefontsize=16, fontfamily="Computer Modern")

In [ ]:
plot_energy(test1_res)

## Test 2: Outside Training Region

In [ ]:
test2_res = ep_analyze(burger, ops, op_fom_tr, Vrmax, Xtest_out, IC_test_out, rmin, rmax, DS)
results_table(test2_res, :SE)

### Results

In [ ]:
p = plot_state_errors!(plot(), test2_res.SE)
plot!(p, yscale=:log10, majorgrid=true, legend=:bottomleft)
xticks!(rmin:rmax)
xlabel!("reduced model dimension " * L"r")
ylabel!("average relative state error")
title!("Test 2 (outside training region)")
plot!(guidefontsize=16, tickfontsize=13, legendfontsize=13, titlefontsize=16, fontfamily="Computer Modern")

In [ ]:
plot_energy(test2_res)

## Plot Test 1 and Test 2 together

In [ ]:
p = plot_state_errors!(plot(), test1_res.SE)
plot_state_errors!(p, test2_res.SE; labels=false)
plot!(p, yscale=:log10, majorgrid=true, legend=:bottomleft)
xticks!(rmin:rmax)
xlabel!("reduced model dimension " * L" r")
ylabel!("average relative state error")
title!("Test", fontsize=18)
ylims!(1e-4, 2)
plot!(guidefontsize=16, tickfontsize=13, legendfontsize=13, titlefontsize=18, fontfamily="Computer Modern")
annotate!(rmax-3.5, 5e-4, text("(1) Interpolation", 13, :left, :bottom, :black, "Computer Modern"))
annotate!(rmax-3.0, 1.5e-1, text("(2) Extrapolation", 13, :left, :top, :black, "Computer Modern"))

## Discussion

With the default sinusoidal initial conditions:

- **Energy preservation**: the least-squares OpInf operators violate the energy-preserving constraint more and more as $r$ grows (residual $\sim 10^{3}$ at $r = 10$), while EPHEC-OpInf satisfies it to machine precision, EPP-OpInf to $\sim 10^{-7}$ (controlled by the penalty weight $\alpha$), and EPSIC-OpInf to within its tolerance $\epsilon$ (on the order of the intrusive model's residual, since $\epsilon$ bounds each of the $\mathcal{O}(r^3)$ constraint terms).
- **Accuracy**: all models are stable for this problem and their state errors are very close to each other. The EP-OpInf models are slightly more accurate than least-squares OpInf at the larger dimensions, i.e., enforcing the physical structure does not cost accuracy.
- **Test 2**: the initial conditions outside the training region have frequencies $\lceil b \rceil \in \{4, \ldots, 7\}$, which do not appear in the training data (frequencies 1–3) and are poorly represented by the POD basis, so all models (including the intrusive one) have large errors there; this is a limitation of the basis, not of the inferred operators.

## Save Results

Save the bases, the reduced operators, and the analysis results (the trajectories are not saved since they can be regenerated).

In [ ]:
mkpath("data")
jldsave(filename;
    IC_FAMILY = string(IC_FAMILY), PAPER_SCALE, rmin, rmax, DS,
    Vrmax, Σr, IC_train, IC_test_in, IC_test_out,
    op_LS, op_int, op_ephec, op_epsic, op_epp,
    train_res, test1_res, test2_res,
)
@info "Results saved to $(filename)"